In [0]:
## 

import csv

file_path = '/Workspace/Users/baodi.wilkinson.external@atradius.com/SAP_BO_Analytics/Rivendell_Analysis/Rivendell_scope_kpi_driven_scope_vs_bo.csv'

csv_content: list = []
unique_report_ids: list = []
with open(file_path, mode='r', encoding='utf-8') as f:
    header: str = f.readline()
    for line in f.readlines():
        line_content = line.strip().split(";")
        csv_row: dict = {
            "rivendell_ref": line_content[0].split("/"),
            "report_id": line_content[1],
            "Business_Unit": "unknown", 
            "report_owner": "unkonwn",
            "report_name": "unknown",
            "cluster_id": -1, 
            "cluster_name": "unknown"        }
        if csv_row["report_id"] not in unique_report_ids:
            unique_report_ids.append(csv_row["report_id"])
            csv_content.append(csv_row)
    print(len(csv_content))


In [0]:
print(csv_content[0])

In [0]:
%sql
select distinct Document_Id, Document_name, UPPER(TRIM(
            CASE WHEN UPPER(TRIM(cms1.created)) = 'ADMINISTRATOR' THEN cms1.lastAuthor ELSE cms1.created END
        )) as owner,
--  created, lastAuthor, 
 Full_path
 from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms as cms1
 left join dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile up1
        ON UPPER(TRIM(up1.UserName)) = UPPER(TRIM(
            CASE WHEN UPPER(TRIM(cms1.created)) = 'ADMINISTRATOR' THEN cms1.lastAuthor ELSE cms1.created END
        ))
 limit 100

In [0]:
from pyspark.sql import functions as F

CLUSTER_NAMES = {
    0: 'Commercial & Business Operations',
    1: 'Financial Reporting & Movement Analysis',
    2: 'Declarations & Policy Group KPI Monitoring',
    3: 'Credit Limits & Engagements Management',
    4: 'Claims Management & Loss Reporting',
    5: 'Core Reference Data & Back-Office Operations',
    6: 'Policy Details, Contracts & Customer Portfolio Analytics',
    7: 'User Access, Authorizations & Sales Pipeline',
    8: 'Global Policy Portfolio & Administration',
    9: 'Premium, Tax & Invoicing Automation',
}

# Build CASE WHEN expression for Spark
cluster_name_expr = F.when(F.col('Cluster_id').isNull(), F.lit('Unclustered'))
for cid, cname in CLUSTER_NAMES.items():
    cluster_name_expr = cluster_name_expr.when(F.col('Cluster_id') == cid, F.lit(cname))
cluster_name_expr = cluster_name_expr.otherwise(F.lit('Unknown'))

df = spark.sql("""
    SELECT DISTINCT
        cms1.Document_Id,
        cms1.Document_name,
        UPPER(TRIM(
            CASE WHEN UPPER(TRIM(cms1.created)) = 'ADMINISTRATOR' THEN cms1.lastAuthor ELSE cms1.created END
        )) AS owner_id,
        -- upper(trim(up1.DisplayName)) as owner_name,
        cms1.Full_path,
        up1.BU as Business_Unit,
        wb1.`cluster` as Cluster_id
    FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms AS cms1
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile up1
        ON UPPER(TRIM(up1.UserName)) = UPPER(TRIM(
            CASE WHEN UPPER(TRIM(cms1.created)) = 'ADMINISTRATOR' THEN cms1.lastAuthor ELSE cms1.created END
        ))
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_cluster_details wb1
        ON upper(trim(wb1.Document_Id)) = upper(trim(cms1.Document_Id))
""").withColumn('Cluster_name', cluster_name_expr)

print(f"Rows: {df.count():,} | Columns: {len(df.columns)}")

# --- Cluster summary ---
cluster_summary = (
    df.groupBy('Cluster_id', 'Cluster_name')
    .agg({'Document_Id': 'count'})
    .withColumnRenamed('count(Document_Id)', 'Document_Count')
    .orderBy('Cluster_id')
    .toPandas()
)

print(f"\nUnique clusters: {cluster_summary['Cluster_id'].nunique()}")
print(f"{'Cluster_id':<6} {'Cluster_name':<55} {'Documents':>10}")
print("-" * 74)
for _, row in cluster_summary.iterrows():
    cid = str(int(row['Cluster_id'])) if row['Cluster_id'] is not None and str(row['Cluster_id']) != 'nan' else 'None'
    print(f"{cid:<6} {row['Cluster_name']:<55} {int(row['Document_Count']):>10,}")
print("-" * 74)
print(f"{'TOTAL':<62} {int(cluster_summary['Document_Count'].sum()):>10,}")

display(df.limit(100))

In [0]:

reports_entry_points: dict = {
    "Brokers": [
        "Broker", 
        "Brokerage",
        "Premium",
        "APR",
        "commission",
        "fees"
    ],
    "TPE": [
        "TPE",
        "Exposure",
        "ECAP",
        "Risk",
        "Rating",
        "TO Ratio",
        "efficiency rate",
        "policy year ratio"
    ],
    "Portfolio": [
        "sale",
        "portfolio",
        "account manager",
        "premium",
        "pipeline",
        "turnover",
        "churn",
        "renewal",
        "performance",
        "PF",
        "cancellation",
        "additionality",
        "AM performance",
        "AM",
        "APR",
        "customer inflow",
        "P&L",
        "profit & loss",
        "profit&loss",
        "profit/loss",
        "profitloss",
        "NB deals"
    ],
    "Claims": [
        "claim",
        "NNP",
        "paid",
        "case",
        "recover"
    ],
    "MMR": [
        "manager",
        "MMR",
        "management",
        "pipeline",
        "country",
        "region"
    ],
    "Non_Rivendell":[]
}



display(df)

In [0]:
import re

# --- Build tagging function from reports_entry_points (cell 5) ---
# MMR has no keywords -> catch-all for reports matching nothing else
KEYWORD_TAGS = {tag: kws for tag, kws in reports_entry_points.items() if kws}
CATCH_ALL_TAG = 'Non_Rivendell'

def tag_report(doc_name, full_path):
    """Return list of matching tags based on Document_name and Full_path."""
    text = ' '.join(filter(None, [str(doc_name or ''), str(full_path or '')])).lower()
    matched = [tag for tag, kws in KEYWORD_TAGS.items()
               if any(re.search(r'\b' + re.escape(kw.lower()) + r'\b', text) for kw in kws)]
    return matched if matched else [CATCH_ALL_TAG]

# --- Apply to df (toPandas since ~22K distinct docs — manageable) ---
pdf = df.toPandas()
pdf['tags'] = pdf.apply(lambda r: tag_report(r['Document_name'], r['Full_path']), axis=1)
pdf['tags_str'] = pdf['tags'].apply(lambda t: ', '.join(t))

# --- Summary ---
from collections import Counter
all_tags = [t for tags in pdf['tags'] for t in tags]
tag_counts = Counter(all_tags)

print(f"Total reports: {len(pdf):,}")
print(f"Reports with multiple tags: {(pdf['tags'].apply(len) > 1).sum():,}")
print("\nTag distribution:")
for tag in ['Brokers', 'TPE', 'Portfolio', 'Claims', 'MMR']:
    print(f"  {tag:<12} {tag_counts.get(tag, 0):>5}")

display(pdf.sort_values('tags_str'))

In [0]:
# --- Load ad_user_profiles for owner name lookup ---
pdf_ad = spark.sql("""
    SELECT UPPER(TRIM(user_ID)) AS user_ID, full_name as DisplayName, BU
    FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles
    WHERE user_ID IS NOT NULL
""").toPandas()

pdf_bo = spark.sql("""
    SELECT UPPER(TRIM(UserName)) AS user_ID, DisplayName, BU
    FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile
    WHERE UserName IS NOT NULL
""").toPandas()

# Merge: bo_user_profile first, fall back to ad_user_profiles
name_lookup = (
    pdf_bo[['user_ID', 'DisplayName']]
    .merge(pdf_ad[['user_ID', 'DisplayName']], on='user_ID', how='outer', suffixes=('_bo', '_ad'))
)
name_lookup['owner_name'] = name_lookup['DisplayName_bo'].combine_first(name_lookup['DisplayName_ad'])
name_lookup = name_lookup[['user_ID', 'owner_name']].drop_duplicates('user_ID')

# --- Enrich pdf with owner_name ---
pdf_enriched = pdf.copy()
pdf_enriched['owner_id_upper'] = pdf_enriched['owner_id'].str.upper().str.strip()
pdf_enriched = pdf_enriched.merge(name_lookup, left_on='owner_id_upper', right_on='user_ID', how='left')
pdf_enriched = pdf_enriched.drop(columns=['owner_id_upper', 'user_ID'])
spark.createDataFrame(pdf_enriched).write.mode('overwrite').saveAsTable('dataplatform01_central_dev_catalog_01.custom_sap_bo.rivendel_tag_inscope_reports')

# # --- Show records still unresolved ---
# unresolved = pdf_enriched[
#     pdf_enriched['owner_name'].isna() |
#     (pdf_enriched['owner_name'].str.strip() == '')
# ].copy()

# print(f"Total reports:      {len(pdf_enriched):,}")
# print(f"Owner name found:   {pdf_enriched['owner_name'].notna().sum():,}")
# print(f"Still unresolved:   {len(unresolved):,} ({len(unresolved)/len(pdf_enriched)*100:.1f}%)")

# if not unresolved.empty:
#     print(f"\nTop unresolved owner_ids:")
#     print(unresolved['owner_id'].value_counts().head(20).to_string())

# display(unresolved[['Document_Id', 'Document_name', 'owner_id', 'Full_path', 'Cluster_name']].sort_values('owner_id'))

In [0]:
%sql
select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.rivendel_tag_inscope_reports

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# --- Cluster name mapping ---
CLUSTER_NAMES = {
    0: 'Commercial & Business Operations',
    1: 'Financial Reporting & Movement Analysis',
    2: 'Declarations & Policy Group KPI Monitoring',
    3: 'Credit Limits & Engagements Management',
    4: 'Claims Management & Loss Reporting',
    5: 'Core Reference Data & Back-Office Operations',
    6: 'Policy Details, Contracts & Customer Portfolio Analytics',
    7: 'User Access, Authorizations & Sales Pipeline',
    8: 'Global Policy Portfolio & Administration',
    9: 'Premium, Tax & Invoicing Automation',
}

# --- BU Group mapping ---
BU_GROUPS = {
    'COMMERCIALS': [
        'COMCE-GERMANY,AUSTRIA, SWITZERL,CEE', 'COMAS-ASIA', 'GLOBA-GLOBAL',
        'COMNA-NORTH AMERICA', 'COMNN-NETHERLANDS', 'COMON-OCEANIA',
        'COMSE-BELGIUM, LUXEMBOURG', 'COMSPB-SPAIN, PORTUGAL, BRAZIL',
        'COMUI-UK & IRELAND', 'FRANC-FRANCE', 'ITALY-ITALY', 'CREDIT INSURANCE'
    ],
    'RISK': [
        'RISK1-RS1-GERMANY, CENTRAL & EAST EUROPE', 'RISK2-RS2-BELGIUM, LUX, FRANCE & ITA',
        'RISK3-RS3-NLD, APAC & AIM', 'RISK4-RS4-NORTH EUROPE, CIS & ACS',
        'RISK4-RS4-NORTH EUROPE, CIS & SP', 'RISK5-RS5-NAFTA',
        'RISK1-RS1-DEU, AUT and CHE', 'RISK1-RS1-Europe, Russia & CIS',
        'RISK2-RS2-NLD, Belux, FRA, Africa & ISR', 'RISK3-RS3-Asia and Oceania',
        'RISK7-RS7-Spain, Portugal, Andorra'
    ],
    'FINANCE': [
        'FINPM-FINANCE PROGRAM MANAGEMENT', 'GFO-GROUP FINANCE OPERATIONS',
        'GFC-GROUP FINANCE AND CONTROL', 'COFIN-CORPORATE FINANCE & TAX',
        'Group Finance', 'Finance', 'Finance & Control'
    ],
}
bu_to_group = {bu.upper().strip(): grp for grp, bus in BU_GROUPS.items() for bu in bus}

def map_bu_group(bu):
    if pd.isna(bu) or str(bu).strip().upper() in ('', 'UNIDENTIFIED', 'LEFT ORGANIZATION'):
        return 'Unidentified'
    return bu_to_group.get(str(bu).upper().strip(), str(bu).strip())

# --- Load data ---
pdf_src = spark.sql("""
    SELECT Document_Id, Cluster_name, Business_Unit, tags
    FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.rivendel_tag_inscope_reports
""").toPandas()

pdf_src['BU_Group'] = pdf_src['Business_Unit'].apply(map_bu_group)

cluster_order = list(CLUSTER_NAMES.values()) + ['Unclustered', 'Unknown']

def build_pivot(df, group_col):
    agg = (
        df.groupby(['Cluster_name', group_col])['Document_Id']
        .nunique().reset_index().rename(columns={'Document_Id': 'cnt'})
    )
    pivot = agg.pivot(index='Cluster_name', columns=group_col, values='cnt').fillna(0)
    ordered = [c for c in cluster_order if c in pivot.index]
    return pivot.reindex(ordered), agg

def stacked_bar(pivot, cols, color_map, title, legend_title):
    fig, ax = plt.subplots(figsize=(16, 7))
    x = np.arange(len(pivot.index))
    bottom = np.zeros(len(pivot))
    for col in cols:
        vals = pivot[col].values
        ax.bar(x, vals, width=0.65, bottom=bottom,
               color=color_map[col], label=col, edgecolor='white', linewidth=0.4)
        bottom += vals
    for i, total in enumerate(bottom):
        if total > 0:
            ax.text(x[i], total + bottom.max() * 0.01, f'{int(total):,}',
                    ha='center', va='bottom', fontsize=8, fontweight='bold', color='#333333')
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index, rotation=30, ha='right', fontsize=9)
    ax.set_xlabel('Cluster', fontsize=10)
    ax.set_ylabel('Distinct Documents', fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylim(0, bottom.max() * 1.15)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(title=legend_title, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=7, title_fontsize=8)
    plt.tight_layout()
    plt.show()

# ── Chart 1: by BU Group ─────────────────────────────────────────────────
pivot1, _ = build_pivot(pdf_src, 'BU_Group')
defined = ['COMMERCIALS', 'RISK', 'FINANCE']
base_colors = plt.cm.tab20.colors
other1 = sorted([c for c in pivot1.columns if c not in defined and c != 'Unidentified'])
cols1 = [c for c in defined if c in pivot1.columns] + other1 + \
        (['Unidentified'] if 'Unidentified' in pivot1.columns else [])
pivot1 = pivot1[cols1]
colors1 = {'COMMERCIALS': '#336699', 'RISK': '#E07B39', 'FINANCE': '#4CAF50', 'Unidentified': '#BEBEBE'}
for i, c in enumerate(c for c in cols1 if c not in colors1):
    colors1[c] = base_colors[i % len(base_colors)]

stacked_bar(pivot1, cols1, colors1, 'In-Scope Reports by Cluster & BU Group', 'BU Group')

# ── Chart 2: by Tag (explode multi-tag rows) ─────────────────────────────────
pdf_exploded = pdf_src.copy()
pdf_exploded['tag'] = pdf_exploded['tags'].apply(
    lambda t: t if isinstance(t, list) else [str(t)]
)
pdf_exploded = pdf_exploded.explode('tag')
pdf_exploded['tag'] = pdf_exploded['tag'].fillna('Non_Rivendell').str.strip()

tag_order = ['Brokers', 'TPE', 'Portfolio', 'Claims', 'MMR', 'Non_Rivendell']
pivot2, _ = build_pivot(pdf_exploded, 'tag')
cols2 = [t for t in tag_order if t in pivot2.columns] + \
        [c for c in pivot2.columns if c not in tag_order]
pivot2 = pivot2[[c for c in cols2 if c in pivot2.columns]]
tag_palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
colors2 = {t: tag_palette[i % len(tag_palette)] for i, t in enumerate(cols2)}

stacked_bar(pivot2, cols2, colors2, 'In-Scope Reports by Cluster & Tag', 'Tag')

print(f"Total distinct documents: {pdf_src['Document_Id'].nunique():,}")
print(f"Total exploded rows (doc x tag): {len(pdf_exploded):,}")

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

CLUSTER_NAMES = {
    0: 'Commercial & Business Operations',
    1: 'Financial Reporting & Movement Analysis',
    2: 'Declarations & Policy Group KPI Monitoring',
    3: 'Credit Limits & Engagements Management',
    4: 'Claims Management & Loss Reporting',
    5: 'Core Reference Data & Back-Office Operations',
    6: 'Policy Details, Contracts & Customer Portfolio Analytics',
    7: 'User Access, Authorizations & Sales Pipeline',
    8: 'Global Policy Portfolio & Administration',
    9: 'Premium, Tax & Invoicing Automation',
}

BU_GROUPS = {
    'COMMERCIALS': [
        'COMCE-GERMANY,AUSTRIA, SWITZERL,CEE', 'COMAS-ASIA', 'GLOBA-GLOBAL',
        'COMNA-NORTH AMERICA', 'COMNN-NETHERLANDS', 'COMON-OCEANIA',
        'COMSE-BELGIUM, LUXEMBOURG', 'COMSPB-SPAIN, PORTUGAL, BRAZIL',
        'COMUI-UK & IRELAND', 'FRANC-FRANCE', 'ITALY-ITALY', 'CREDIT INSURANCE'
    ],
    'RISK': [
        'RISK1-RS1-GERMANY, CENTRAL & EAST EUROPE', 'RISK2-RS2-BELGIUM, LUX, FRANCE & ITA',
        'RISK3-RS3-NLD, APAC & AIM', 'RISK4-RS4-NORTH EUROPE, CIS & ACS',
        'RISK4-RS4-NORTH EUROPE, CIS & SP', 'RISK5-RS5-NAFTA',
        'RISK1-RS1-DEU, AUT and CHE', 'RISK1-RS1-Europe, Russia & CIS',
        'RISK2-RS2-NLD, Belux, FRA, Africa & ISR', 'RISK3-RS3-Asia and Oceania',
        'RISK7-RS7-Spain, Portugal, Andorra'
    ],
    'FINANCE': [
        'FINPM-FINANCE PROGRAM MANAGEMENT', 'GFO-GROUP FINANCE OPERATIONS',
        'GFC-GROUP FINANCE AND CONTROL', 'COFIN-CORPORATE FINANCE & TAX',
        'Group Finance', 'Finance', 'Finance & Control'
    ],
}
bu_to_group = {bu.upper().strip(): grp for grp, bus in BU_GROUPS.items() for bu in bus}

def map_bu_group(bu):
    if pd.isna(bu) or str(bu).strip().upper() in ('', 'UNIDENTIFIED', 'LEFT ORGANIZATION'):
        return 'Unidentified'
    return bu_to_group.get(str(bu).upper().strip(), str(bu).strip())

DEFINED_GROUPS = ['COMMERCIALS', 'RISK', 'FINANCE']

def bu_group_order(all_bu_groups):
    others = sorted([c for c in all_bu_groups if c not in DEFINED_GROUPS and c != 'Unidentified'])
    return [c for c in DEFINED_GROUPS if c in all_bu_groups] + others + \
           (['Unidentified'] if 'Unidentified' in all_bu_groups else [])

# --- Explode tags: one row per tag ---
df_exploded = spark.sql("""
    SELECT * FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.rivendel_tag_inscope_reports
""").withColumn('tag', F.explode(
    F.when(F.col('tags').isNull(), F.array(F.lit('Non_Rivendell')))
     .otherwise(F.col('tags'))
))

print(f"Original distinct documents : {df_exploded.select('Document_Id').distinct().count():,}")
print(f"Exploded rows (1 tag each)  : {df_exploded.count():,}")

display(df_exploded.orderBy('Document_Id', 'tag'))

# --- Shared config ---
tag_order   = ['Brokers', 'TPE', 'Portfolio', 'Claims', 'MMR', 'Non_Rivendell']
tag_palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

def stacked_bar_tags(ax_df, x_col, x_order, title, y_label='Distinct Documents'):
    cols_in_data = ax_df['tag'].unique().tolist()
    cols  = [t for t in tag_order if t in cols_in_data] + \
            [c for c in cols_in_data if c not in tag_order]
    colors = {t: tag_palette[i % len(tag_palette)] for i, t in enumerate(
        [t for t in tag_order if t in cols_in_data] + [c for c in cols_in_data if c not in tag_order]
    )}

    agg = (
        ax_df.groupby([x_col, 'tag'])['Document_Id']
        .nunique().reset_index().rename(columns={'Document_Id': 'cnt'})
    )
    pivot = agg.pivot(index=x_col, columns='tag', values='cnt').fillna(0)
    pivot = pivot.reindex([c for c in x_order if c in pivot.index])
    pivot = pivot[[c for c in cols if c in pivot.columns]]

    fig, ax = plt.subplots(figsize=(16, 7))
    x      = np.arange(len(pivot.index))
    bottom = np.zeros(len(pivot))
    for col in pivot.columns:
        vals = pivot[col].values
        ax.bar(x, vals, width=0.65, bottom=bottom,
               color=colors[col], label=col, edgecolor='white', linewidth=0.4)
        bottom += vals
    for i, total in enumerate(bottom):
        if total > 0:
            ax.text(x[i], total + bottom.max() * 0.01, f'{int(total):,}',
                    ha='center', va='bottom', fontsize=8, fontweight='bold', color='#333333')
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel(y_label, fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylim(0, bottom.max() * 1.15)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(title='Tag', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8, title_fontsize=9)
    plt.tight_layout()
    plt.show()

pdf_exp = df_exploded.select('Document_Id', 'Cluster_name', 'Business_Unit', 'tag').toPandas()
pdf_exp['BU_Group'] = pdf_exp['Business_Unit'].apply(map_bu_group)
pdf_exp_rivendell = pdf_exp[pdf_exp['tag'] != 'Non_Rivendell'].copy()

cluster_order = list(CLUSTER_NAMES.values()) + ['Unclustered', 'Unknown']

# --- Chart 1: by Cluster (all tags) ---
stacked_bar_tags(
    pdf_exp, 'Cluster_name', cluster_order,
    'In-Scope Reports by Cluster & Tag (exploded)'
)

# --- Chart 2: by Owner BU Group (all tags) ---
bu_order = bu_group_order(pdf_exp['BU_Group'].unique().tolist())
stacked_bar_tags(
    pdf_exp, 'BU_Group', bu_order,
    'In-Scope Reports by Owner BU Group & Tag (exploded)'
)

# --- Chart 3: by Owner BU Group (Rivendell tags only, excl. Non_Rivendell) ---
bu_order_rv = bu_group_order(pdf_exp_rivendell['BU_Group'].unique().tolist())
stacked_bar_tags(
    pdf_exp_rivendell, 'BU_Group', bu_order_rv,
    'In-Scope Reports by Owner BU Group & Tag — Rivendell Tags Only (excl. Non_Rivendell)',
    y_label='Distinct Documents Tagged'
)

# --- Chart 4: by Cluster (Rivendell tags only, excl. Non_Rivendell) ---
stacked_bar_tags(
    pdf_exp_rivendell, 'Cluster_name', cluster_order,
    'In-Scope Reports by Cluster & Tag — Rivendell Tags Only (excl. Non_Rivendell)',
    y_label='Distinct Documents Tagged'
)

print(f"Total distinct documents          : {pdf_exp['Document_Id'].nunique():,}")
print(f"Rivendell-tagged documents        : {pdf_exp_rivendell['Document_Id'].nunique():,}")
print(f"Total exploded rows               : {len(pdf_exp):,}")
print(f"Rivendell-tagged exploded rows    : {len(pdf_exp_rivendell):,}")

In [0]:
multi_tagged = pdf[pdf['tags'].apply(len) > 1].copy()
multi_tagged = multi_tagged.sort_values('tags_str')

print(f"Reports with multiple tags: {len(multi_tagged):,} / {len(pdf):,} ({len(multi_tagged)/len(pdf)*100:.1f}%)")
print(f"\nTag combination breakdown:")
print(multi_tagged['tags_str'].value_counts().to_string())

display(multi_tagged[['Document_Id', 'Document_name', 'Full_path', 'tags_str']])

In [0]:
no_tag = pdf[pdf['tags'].apply(lambda t: t == ['Non_Rivendell'])].copy()

print(f"Reports with no keyword match (Non_Rivendell): {len(no_tag):,} / {len(pdf):,} ({len(no_tag)/len(pdf)*100:.1f}%)")
display(no_tag[['Document_Id', 'Document_name', 'Full_path']].sort_values('Document_name'))

In [0]:
import re
from collections import Counter

pattern = re.compile(r'manager|mgr|mmr|monthly\s*management|management\s*report', re.IGNORECASE)

matches = pdf[
    pdf['Document_name'].str.contains(pattern, na=False) |
    pdf['Full_path'].str.contains(pattern, na=False)
].copy()

print(f"Matching rows: {len(matches):,}\n")

print("=== Unique Document_name values ===")
for v in sorted(matches['Document_name'].dropna().unique()):
    print(f"  {v}")

print(f"\n=== Full_path folder segments containing match (with count) ===")
path_hits = Counter()
for p in matches['Full_path'].dropna():
    for part in p.split('/'):
        if pattern.search(part):
            path_hits[part.strip()] += 1

for seg, cnt in path_hits.most_common():
    print(f"  [{cnt:>4}]  {seg}")